In [59]:
import numpy as np

In [60]:
import pandas as pd

In [61]:
dataset = pd.read_csv("Social_Network_Ads.csv")

In [62]:
dataset

,User ID,Gender,Age,EstimatedSalary,Purchased
0,15624510,Male,19,19000,0
1,15810944,Male,35,20000,0
2,15668575,Female,26,43000,0
3,15603246,Female,27,57000,0
4,15804002,Male,19,76000,0
...,...,...,...,...,...
395,15691863,Female,46,41000,1
396,15706071,Male,51,23000,1
397,15654296,Female,50,20000,1
398,15755018,Male,36,33000,0


In [63]:
dataset = pd.get_dummies(dataset, dtype=int, drop_first = True)

In [64]:
dataset.drop("User ID", axis=1)

,Age,EstimatedSalary,Purchased,Gender_Male
0,19,19000,0,1
1,35,20000,0,1
2,26,43000,0,0
3,27,57000,0,0
4,19,76000,0,1
...,...,...,...,...
395,46,41000,1,0
396,51,23000,1,1
397,50,20000,1,0
398,36,33000,0,1


In [65]:
dataset.columns

Index(['User ID', 'Age', 'EstimatedSalary', 'Purchased', 'Gender_Male'], dtype='object')

In [66]:
independent = dataset[["Age", "EstimatedSalary", "Gender_Male"]]

In [67]:
independent

,Age,EstimatedSalary,Gender_Male
0,19,19000,1
1,35,20000,1
2,26,43000,0
3,27,57000,0
4,19,76000,1
...,...,...,...
395,46,41000,0
396,51,23000,1
397,50,20000,0
398,36,33000,1


In [68]:
dependent = dataset[["Purchased"]]

In [69]:
dependent

,Purchased
0,0
1,0
2,0
3,0
4,0
...,...
395,1
396,1
397,1
398,0


In [70]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(independent, dependent, test_size=0.30, random_state=0)

In [71]:
from sklearn.preprocessing import StandardScaler
sc=StandardScaler()
x_train = sc.fit_transform(x_train)
x_test = sc.transform(x_test)

In [72]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
param_grid = {'criterion':['gini','entropy'],'max_features':['sqrt','log2'],'n_estimators':[10,100]}
grid = GridSearchCV(RandomForestClassifier(), param_grid, refit = True, verbose=3, n_jobs=-1, scoring='f1_weighted')
grid.fit(x_train, np.ravel(y_train))
re=grid.cv_results_
grid_predictions=grid.predict(x_test)

Fitting 5 folds for each of 8 candidates, totalling 40 fits


In [73]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, grid_predictions)

In [74]:
from sklearn.metrics import classification_report
clf_report = classification_report(y_test, grid_predictions)

In [75]:
#from sklearn.metrics import f1_score
#f1_macro = f1_score(y_test, grid_predictions, average = 'weighted')
#print("The f1_macro value for the best parameter{}:".format(grid.best_params_),f1_macro)
print("The best parameter set for this model:\n", format(grid.best_params_))
print("The Confusion matrix:\n", cm)
print("The Classfication report:\n", clf_report)
from sklearn.metrics import roc_auc_score
roc_score = roc_auc_score(y_test, grid.predict_proba(x_test)[:,1])
print("The ROC_AUC for this model is:\n", roc_score)

The best parameter set for this model:
 {'criterion': 'gini', 'max_features': 'log2', 'n_estimators': 100}
The Confusion matrix:
 [[74  5]
 [ 5 36]]
The Classfication report:
               precision    recall  f1-score   support

           0       0.94      0.94      0.94        79
           1       0.88      0.88      0.88        41

    accuracy                           0.92       120
   macro avg       0.91      0.91      0.91       120
weighted avg       0.92      0.92      0.92       120

The ROC_AUC for this model is:
 0.966193269527632


In [76]:
table = pd.DataFrame.from_dict(re)
table

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_criterion,param_max_features,param_n_estimators,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.032069,0.002726,0.006662,0.000396,gini,sqrt,10,"{'criterion': 'gini', 'max_features': 'sqrt', ...",0.855314,0.838326,0.841398,0.892857,0.909115,0.867402,0.028480,7
1,0.257782,0.016841,0.013588,0.000891,gini,sqrt,100,"{'criterion': 'gini', 'max_features': 'sqrt', ...",0.874254,0.838326,0.823129,0.911105,0.946153,0.878593,0.045471,4
2,0.026578,0.002154,0.005413,0.000963,gini,log2,10,"{'criterion': 'gini', 'max_features': 'log2', ...",0.892857,0.840114,0.841398,0.911105,0.890114,0.875118,0.028972,5
3,0.246438,0.006216,0.014180,0.001019,gini,log2,100,"{'criterion': 'gini', 'max_features': 'log2', ...",0.874254,0.875644,0.823129,0.911105,0.982051,0.893237,0.052521,1
4,0.027569,0.001291,0.004875,0.000433,entropy,sqrt,10,"{'criterion': 'entropy', 'max_features': 'sqrt...",0.874254,0.802399,0.804584,0.892857,0.964286,0.867676,0.060420,6
5,0.246912,0.001774,0.013015,0.000239,entropy,sqrt,100,"{'criterion': 'entropy', 'max_features': 'sqrt...",0.874254,0.857143,0.823129,0.911105,0.964286,0.885983,0.048337,3
6,0.026021,0.001616,0.004575,0.000275,entropy,log2,10,"{'criterion': 'entropy', 'max_features': 'log2...",0.835985,0.821429,0.823129,0.892857,0.890114,0.852703,0.032075,8
7,0.219449,0.013257,0.009231,0.001775,entropy,log2,100,"{'criterion': 'entropy', 'max_features': 'log2...",0.874254,0.875644,0.806153,0.911105,0.982051,0.889841,0.057276,2


In [77]:
Age_input = float(input("Age="))
Est_salary_input = float(input("Estimated Salary="))
Gender_male_input = int(input("Gender Male="))

Age= 51
Estimated Salary= 50000
Gender Male= 0


In [80]:
Future_predictions = grid.predict([[Age_input, Est_salary_input, Gender_male_input]])

In [81]:
print("Future predictions:\n", format(Future_predictions))

Future predictions:
 [1]
